In [1]:
import time
import joblib
import warnings
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# Load processor and data
preprocessor = joblib.load("../../artifacts/computer_preprocessor.joblib")
df = pd.read_csv("../../data/processed/computer_prices_final.csv")

TARGET = "price"
raw_or_parse_columns = ["model", "cpu_model", "gpu_model", "resolution"]
X = df.drop(columns=[TARGET] + raw_or_parse_columns)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

X_train = preprocessor.transform(X_train)
X_test = preprocessor.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(max_iter=10000),           # increase iterations to ensure convergence
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,                                # default is 100, keep it
        random_state=42,
        n_jobs=-1,                                       # use all CPU cores
        max_depth=10                                     # limit depth for speed & generalisation
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,                                # default is 100
        random_state=42,
        learning_rate=0.1,
        max_depth=5                                      # limit depth to speed up
    )
    # "SVR (Linear)": SVR(kernel='linear')
}

def evaluate_model(name, model):
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)
    return {
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R² Score": r2,
        "Training Time (sec)": training_time,
        "Model Object": model
    }

results = []
for name, model in models.items():
    print(f"Training {name}...")
    results.append(evaluate_model(name, model))

results_df = pd.DataFrame(results).sort_values(by="R² Score", ascending=False)
print(results_df)

# Save the best model
best_model_name = results_df.iloc[0]["Model"]
best_model = results_df.iloc[0]["Model Object"]
joblib.dump(best_model, "../../artifacts/computer_best_model.joblib")
print(f"Saved best model: {best_model_name}")


Training Linear Regression...
Training Ridge Regression...
Training Lasso Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
               Model         MAE           MSE        RMSE  R² Score  \
5  Gradient Boosting  140.679679  40510.232985  201.271540  0.877321   
1   Ridge Regression  143.099290  41103.142904  202.739101  0.875526   
0  Linear Regression  143.099690  41103.249821  202.739364  0.875525   
2   Lasso Regression  144.432258  41705.484409  204.219207  0.873702   
4      Random Forest  176.683567  58079.636091  240.997170  0.824115   
3      Decision Tree  188.925156  79418.926463  281.813638  0.759493   

   Training Time (sec)                                       Model Object  
5            48.790498  ([DecisionTreeRegressor(max_depth=5,\n        ...  
1             0.085499                                            Ridge()  
0             0.554041                                 LinearRegression()  
2             1.93654